In [13]:
# Task 4.1

import sqlite3

conn = sqlite3.connect("STORE.db")
cursor = conn.cursor()

# Create Donut table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Donut (
        DonutID INTEGER PRIMARY KEY NOT NULL,
        DonutName TEXT NOT NULL,
        UnitPrice REAL NOT NULL
    );
""")

# Create Member table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Member (
        MemberNumber INTEGER PRIMARY KEY NOT NULL,
        MemberName TEXT NOT NULL,
        Phone INTEGER NOT NULL
    );
""")

# Create Sale table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Sale (
        SaleID INTEGER PRIMARY KEY NOT NULL,
        MemberNumber INTEGER NOT NULL,
        DonutID INTEGER NOT NULL,
        Date INTEGER NOT NULL,
        Quantity INTEGER NOT NULL,
        FOREIGN KEY (MemberNumber) REFERENCES Member(MemberNumber),
        FOREIGN KEY (DonutID) REFERENCES Donut(DonutID)
    );
""")

conn.commit()
conn.close()

In [14]:
# Task 4.2

import sqlite3

# Read donut data
donut_file = open("DONUT.txt", "r")
donut_array = []
for line in donut_file:
    donut = line.strip().split(',')
    DonutID = int(donut[0])
    DonutName = donut[1]
    UnitPrice = float(donut[2])
    donut_array.append([DonutID, DonutName, UnitPrice])
donut_file.close()

# Read member data
member_file = open("MEMBER.txt", "r")
member_array = []
for line in member_file:
    member = line.strip().split(',')
    MemberNumber = int(member[0])
    MemberName = member[1]
    Phone = int(member[2])
    member_array.append([MemberNumber, MemberName, Phone])
member_file.close()

# Read sale data
sale_file = open("SALE.txt", "r")
sale_array = []
for line in sale_file:
    sale = line.strip().split(',')
    SaleID = int(sale[0])
    MemberNumber = int(sale[1])
    DonutID = int(sale[2])
    Date = int(sale[3])
    Quantity = int(sale[4])
    sale_array.append([SaleID, MemberNumber, DonutID, Date, Quantity])
sale_file.close()

# Insert into database
conn = sqlite3.connect("STORE.db")
cursor = conn.cursor()

# Insert donuts
for donut in donut_array:
    cursor.execute("""
        INSERT INTO Donut (DonutID, DonutName, UnitPrice)
        VALUES (?, ?, ?)
    """, donut)

# Insert members
for member in member_array:
    cursor.execute("""
        INSERT INTO Member (MemberNumber, MemberName, Phone)
        VALUES (?, ?, ?)
    """, member)

# Insert sales
for sale in sale_array:
    cursor.execute("""
        INSERT INTO Sale (SaleID, MemberNumber, DonutID, Date, Quantity)
        VALUES (?, ?, ?, ?, ?)
    """, sale)

conn.commit()
conn.close()

In [22]:
# Task 4.3

import sqlite3

def get_member():
    try:
        MemberNumber = int(input("Enter number of member: "))
    except ValueError:
        print("Invalid input. Please enter a number.")
        return

    conn = sqlite3.connect("STORE.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT m.MemberName, d.DonutName, s.Date, s.Quantity
        FROM Member AS m
        INNER JOIN Sale AS s ON m.MemberNumber = s.MemberNumber
        INNER JOIN Donut AS d ON s.DonutID = d.DonutID
        WHERE m.MemberNumber = ?
    """, (MemberNumber, ))

    rows = cursor.fetchall()

    if not rows:
        print("No sales found for this member.")
        return

    member_name = rows[0][0]
    print(f"\nSales for member: {member_name} (Member Number: {MemberNumber})")
    print("-" * 50)
    print(f"{'Donut Name':20} {'Date':10} {'Quantity':8}")
    print("-" * 50)

    for row in rows:
        donut_name = row[1]
        date = row[2]
        quantity = row[3]
        print(f"{donut_name:20} {date:<10} {quantity:<8}")

    conn.close()

get_member()


Enter number of member:  102



Sales for member: Ben (Member Number: 102)
--------------------------------------------------
Donut Name           Date       Quantity
--------------------------------------------------
Honey Fashion        20230722   3       
Ping Straberry       20230722   3       
